# 02 — Preprocesamiento y Feature Engineering

Verifica el pipeline de normalización y genera features.


In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from discurso_odio.config import load_settings
from discurso_odio.data.loaders import load_all_splits, load_split, get_class_weights
from discurso_odio.paths import REPORTS_FIGURES, REPORTS_METRICS
from discurso_odio.utils.plotting import setup_plot_style, get_label_names, get_class_palette
from discurso_odio.utils.seed import set_seed

settings = load_settings()
set_seed(settings['project']['seed'])
setup_plot_style()
LABELS = get_label_names()
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)
REPORTS_METRICS.mkdir(parents=True, exist_ok=True)
print('Proyecto:', settings['project']['name'])

from discurso_odio.features.normalization import (
    normalize_for_model, has_accents_or_enye, has_uppercase,
    has_punctuation, has_mention_literal, has_url_literal, has_emoji_raw,
)
from discurso_odio.features.text_features import build_text_features



Proyecto: discurso-odio-sv


In [2]:
corpus = load_split('corpus')
train = load_split('train')
print(len(corpus), 'registros')


3063

 registros


## Verificación del pipeline de normalización


In [3]:
orig = corpus['texto_original']
modelo = corpus['texto_modelo']
verification = pd.DataFrame({
    'propiedad': ['tildes/ñ', 'mayúsculas', 'puntuación', 'mención literal', 'URL literal', 'emoji crudo'],
    'texto_original': [
        orig.apply(has_accents_or_enye).sum(),
        orig.apply(has_uppercase).sum(),
        orig.apply(has_punctuation).sum(),
        orig.apply(has_mention_literal).sum(),
        orig.apply(has_url_literal).sum(),
        orig.apply(has_emoji_raw).sum(),
    ],
    'texto_modelo': [
        modelo.apply(has_accents_or_enye).sum(),
        modelo.apply(has_uppercase).sum(),
        modelo.apply(has_punctuation).sum(),
        modelo.apply(lambda t: '@usuario' in str(t)).sum(),
        modelo.apply(lambda t: 'http://url' in str(t)).sum(),
        modelo.apply(has_emoji_raw).sum(),
    ],
})
verification


,propiedad,texto_original,texto_modelo
0,tildes/ñ,1797,1827
1,mayúsculas,2886,2880
2,puntuación,2053,2053
3,mención literal,147,147
4,URL literal,5,2
5,emoji crudo,193,0


## Feature engineering


In [4]:
features_df = build_text_features(corpus)
feature_cols = [c for c in features_df.columns if c.startswith(('n_', 'upper_', 'exclamation', 'question', 'has_', 'es_', 'sarcasmo', 'ironia', 'contexto', 'dialecto'))]
features_df[feature_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
n_chars,3063.0,115.336925,103.371806,7.0,49.000000,88.000000,150.500000,1500.000000
n_palabras,3063.0,20.193928,18.038503,1.0,9.000000,16.000000,26.000000,231.000000
n_emojis,3063.0,0.159647,0.903389,0.0,0.000000,0.000000,0.000000,22.000000
n_menciones,3063.0,0.079660,0.429392,0.0,0.000000,0.000000,0.000000,6.000000
n_hashtags,3063.0,0.230167,0.890243,0.0,0.000000,0.000000,0.000000,14.000000
n_jerga,3063.0,1.095005,1.142858,0.0,0.000000,1.000000,2.000000,7.000000
upper_ratio,3063.0,0.062518,0.139684,0.0,0.012821,0.024272,0.047619,0.965517
exclamation_count,3063.0,0.197519,0.830018,0.0,0.000000,0.000000,0.000000,12.000000
question_count,3063.0,0.266405,1.113220,0.0,0.000000,0.000000,0.000000,18.000000
has_url,3063.0,0.001632,0.040376,0.0,0.000000,0.000000,0.000000,1.000000


## Pesos de clase (solo train)


In [5]:
weights = get_class_weights(train)
expected = {3: 1.3779, 2: 1.2099, 0: 0.9404, 1: 0.7224}
comparison = pd.DataFrame({
    'clase': [LABELS[i] for i in sorted(weights)],
    'peso_calculado': [round(weights[i], 4) for i in sorted(weights)],
    'peso_informe': [expected[i] for i in sorted(weights)],
})
comparison


,clase,peso_calculado,peso_informe
0,No Tóxico,0.9404,0.9404
1,Lenguaje Ofensivo,0.7224,0.7224
2,Discurso de Odio,1.2099,1.2099
3,Amenazas/Violencia,1.3779,1.3779


In [6]:
comparison.to_csv(REPORTS_METRICS / 'class_weights.csv', index=False)
print('Pesos exportados.')


Pesos exportados.
